# Phase 1: 数据探索 & 特征工程

本 notebook 覆盖 Phase 1 的四个任务：
- **Task A**: 停电数据探索（目标变量）
- **Task B**: 天气特征探索 & 特征筛选
- **Task C**: 通用特征构造
- **Task D**: Baseline 复现 & 评估框架

每个人负责自己的部分，但建议大家都跑一遍，了解全貌。

In [ ]:
# ============== 安装 & 导入 ==============
# Colab 用户取消下面的注释
# !pip install netCDF4 xarray

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# ====== 中文字体配置 ======
import matplotlib
import platform
if platform.system() == 'Darwin':  # macOS
    matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti']
elif platform.system() == 'Linux':  # Linux / Colab
    matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'SimHei']
else:  # Windows
    matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 数据路径 — 确保 train.nc 已经放到 data/ 文件夹
DATA_DIR = "data/"
ds = xr.open_dataset(os.path.join(DATA_DIR, "train.nc"))

# 基本信息提取
timestamps = pd.to_datetime(ds.timestamp.values)
locations = list(ds.location.values)
weather_features = list(ds.feature.values) if 'feature' in ds.dims else []

# 核心数据矩阵
outage = ds.out.transpose("timestamp", "location").values.astype(float)      # (T, L)
tracked = ds.tracked.transpose("timestamp", "location").values.astype(float)  # (T, L)
weather = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)  # (T, L, F)

T, L = outage.shape
F = len(weather_features)

print(f"时间范围: {timestamps.min()} ~ {timestamps.max()}")
print(f"时间步数: {T} (小时)")
print(f"县的数量: {L}")
print(f"天气特征数: {F}")
print(f"停电数据 shape: {outage.shape}")
print(f"天气数据 shape: {weather.shape}")

---
# Task A: 停电数据探索

**目标**：搞清楚停电数据长什么样——分布、哪些县严重、有没有时间规律。

## A1: 停电数据的整体分布

先看看 `out` 这个变量长什么样。剧透：绝大多数时间没停电（=0），偶尔有极端大值。

In [ ]:
# A1: 停电数据的基本统计
flat_out = outage.flatten()
print("=== 停电数据基本统计 ===")
print(f"总观测数: {len(flat_out):,}")
print(f"零值占比: {(flat_out == 0).mean():.1%}")
print(f"均值: {flat_out.mean():.2f}")
print(f"中位数: {np.median(flat_out):.2f}")
print(f"标准差: {flat_out.std():.2f}")
print(f"最大值: {flat_out.max():.0f}")
print(f"99%分位: {np.percentile(flat_out, 99):.0f}")
print(f"99.9%分位: {np.percentile(flat_out, 99.9):.0f}")

# 直方图：排除零值看分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：全部数据
axes[0].hist(flat_out, bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title("停电数分布 (全部数据)")
axes[0].set_xlabel("停电户数")
axes[0].set_ylabel("频次")
axes[0].set_yscale('log')

# 右图：只看非零值
nonzero = flat_out[flat_out > 0]
axes[1].hist(nonzero, bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title(f"停电数分布 (仅非零值, n={len(nonzero):,})")
axes[1].set_xlabel("停电户数")
axes[1].set_ylabel("频次")
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n结论: 数据极度右偏，{(flat_out==0).mean():.0%} 的观测值为0，极端值可达 {flat_out.max():.0f}。")
print("RMSE 对大值非常敏感，所以准确预测极端停电事件是拿分关键。")

## A2: 各县停电排名

哪些县停电最严重？从总量和峰值两个角度看。这直接关系到 Part II 选哪几个县放发电机。

In [ ]:
# A2: 各县停电统计
county_stats = pd.DataFrame({
    'location': locations,
    'total_outage': outage.sum(axis=0),
    'mean_outage': outage.mean(axis=0),
    'max_outage': outage.max(axis=0),
    'mean_tracked': tracked.mean(axis=0),
})
# 停电率 = 平均停电数 / 平均追踪用户数
county_stats['outage_rate'] = county_stats['mean_outage'] / county_stats['mean_tracked'].replace(0, np.nan)

# Top 10 按总停电量
top10_total = county_stats.nlargest(10, 'total_outage')
# Top 10 按停电率
top10_rate = county_stats.dropna(subset=['outage_rate']).nlargest(10, 'outage_rate')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 总停电量
axes[0].barh(top10_total['location'].astype(str), top10_total['total_outage'], color='steelblue')
axes[0].set_xlabel('总停电户数 (累计)')
axes[0].set_title('Top 10 县 (按总停电量)')
axes[0].invert_yaxis()

# 峰值停电
top10_max = county_stats.nlargest(10, 'max_outage')
axes[1].barh(top10_max['location'].astype(str), top10_max['max_outage'], color='coral')
axes[1].set_xlabel('单小时最大停电户数')
axes[1].set_title('Top 10 县 (按峰值停电)')
axes[1].invert_yaxis()

# 停电率
axes[2].barh(top10_rate['location'].astype(str), top10_rate['outage_rate'], color='seagreen')
axes[2].set_xlabel('平均停电率 (out/tracked)')
axes[2].set_title('Top 10 县 (按停电率)')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print("=== Top 10 县详情 (按总停电量) ===")
print(top10_total[['location', 'total_outage', 'max_outage', 'mean_tracked', 'outage_rate']].to_string(index=False))

## A3: 停电的时间规律

按小时、按天看停电是否有周期性模式。

In [ ]:
# A3: 时间模式分析
total_outage_by_time = outage.sum(axis=1)  # 每小时全州总停电

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# 1) 全时段总停电趋势
axes[0].plot(timestamps, total_outage_by_time, linewidth=0.8)
axes[0].set_title("全州每小时总停电数 (时间趋势)")
axes[0].set_xlabel("时间")
axes[0].set_ylabel("停电户数")
# 标出极端事件
peak_idx = np.argsort(total_outage_by_time)[-5:]  # top 5 peaks
for idx in peak_idx:
    axes[0].annotate(f"{timestamps[idx].strftime('%m/%d %Hh')}\n{total_outage_by_time[idx]:.0f}",
                     xy=(timestamps[idx], total_outage_by_time[idx]),
                     fontsize=8, color='red', ha='center')

# 2) 按小时聚合（日内模式）
hour_of_day = timestamps.hour
hourly_avg = pd.Series(total_outage_by_time).groupby(hour_of_day).mean()
axes[1].bar(hourly_avg.index, hourly_avg.values, color='steelblue', alpha=0.8)
axes[1].set_title("按小时聚合的平均停电数 (日内模式)")
axes[1].set_xlabel("小时 (0-23)")
axes[1].set_ylabel("平均停电户数")
axes[1].set_xticks(range(24))

# 3) 按天聚合
daily_total = pd.Series(total_outage_by_time, index=timestamps).resample('D').sum()
axes[2].bar(daily_total.index, daily_total.values, color='coral', alpha=0.8)
axes[2].set_title("每日总停电数")
axes[2].set_xlabel("日期")
axes[2].set_ylabel("停电户数")
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 找出最严重的几天
top_days = daily_total.nlargest(5)
print("=== 停电最严重的 5 天 ===")
for date, val in top_days.items():
    print(f"  {date.strftime('%Y-%m-%d')}: {val:,.0f} 户")

---
# Task B: 天气特征探索 & 特征筛选

**目标**：从 109 个天气变量里找出和停电最相关的，删掉冗余的，缩减到 15-25 个。

## B1: 天气特征与停电的相关性排名

把每个天气变量的全部数据展平，算和停电的 Pearson 相关系数，排序找 top 20。

In [ ]:
# B1: 各天气变量与停电的相关性
flat_out = outage.flatten()
correlations = {}

for fi, feat_name in enumerate(weather_features):
    flat_w = weather[:, :, fi].flatten()
    # 去掉 NaN
    mask = ~(np.isnan(flat_out) | np.isnan(flat_w))
    if mask.sum() > 100:
        correlations[feat_name] = np.corrcoef(flat_out[mask], flat_w[mask])[0, 1]

corr_series = pd.Series(correlations).sort_values(key=abs, ascending=False)

# 画 top 30
fig, ax = plt.subplots(figsize=(10, 10))
top30 = corr_series.head(30)
colors = ['coral' if v > 0 else 'steelblue' for v in top30.values]
ax.barh(range(len(top30)), top30.values, color=colors)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30.index)
ax.set_xlabel('Pearson 相关系数')
ax.set_title('天气特征与停电的相关性 (Top 30)')
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("=== Top 20 相关特征 ===")
for name, corr in corr_series.head(20).items():
    print(f"  {name:15s}: {corr:+.4f}")

print(f"\n=== 相关性很弱的特征 (|r| < 0.01, 候选删除) ===")
weak = corr_series[corr_series.abs() < 0.01]
print(f"  共 {len(weak)} 个: {list(weak.index)}")

## B2: 特征之间的共线性分析

如果两个天气变量相关性 > 0.9，说明它们几乎是同一个东西，留一个就行。

In [ ]:
# B2: Top 特征之间的共线性热力图
# 只看与停电相关性 top 25 的特征，避免热力图太大
top25_names = list(corr_series.head(25).index)
top25_indices = [weather_features.index(n) for n in top25_names]

# 用全局数据算特征间相关矩阵
weather_flat = weather.reshape(-1, F)  # (T*L, F)
top25_data = weather_flat[:, top25_indices]
feat_corr_matrix = np.corrcoef(top25_data.T)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(feat_corr_matrix, xticklabels=top25_names, yticklabels=top25_names,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=False, ax=ax)
ax.set_title("Top 25 天气特征之间的相关性矩阵")
plt.tight_layout()
plt.show()

# 找出高度共线性的特征对 (|r| > 0.9)
print("=== 高度共线性的特征对 (|r| > 0.9) ===")
print("可以考虑每对只保留一个（保留与停电相关性更强的那个）\n")
seen = set()
for i in range(len(top25_names)):
    for j in range(i+1, len(top25_names)):
        r = feat_corr_matrix[i, j]
        if abs(r) > 0.9:
            pair = f"  {top25_names[i]:15s} <-> {top25_names[j]:15s}: r={r:.3f}"
            print(pair)

## B3: 关键天气变量 vs 停电的时序对比

选几个最相关的天气变量，看它们在极端停电事件前后的变化。

In [ ]:
# B3: 关键天气变量 vs 全州停电的时序对比
# 选前 6 个最相关的特征
top6_feats = list(corr_series.head(6).index)

fig, axes = plt.subplots(len(top6_feats) + 1, 1, figsize=(16, 3 * (len(top6_feats) + 1)), sharex=True)

# 第一行：停电总数
axes[0].plot(timestamps, total_outage_by_time, color='black', linewidth=0.8)
axes[0].set_ylabel("停电总数")
axes[0].set_title("全州停电 vs 关键天气变量")

# 后续行：各天气变量（全县平均）
for i, feat_name in enumerate(top6_feats):
    fi = weather_features.index(feat_name)
    feat_avg = weather[:, :, fi].mean(axis=1)  # 全县平均
    corr_val = corr_series[feat_name]
    axes[i+1].plot(timestamps, feat_avg, linewidth=0.8, color='steelblue')
    axes[i+1].set_ylabel(feat_name)
    axes[i+1].set_title(f"{feat_name} (r={corr_val:+.3f})", loc='left', fontsize=10)

axes[-1].set_xlabel("时间")
plt.tight_layout()
plt.show()

## B4: 缺失值检查

In [ ]:
# B4: 缺失值检查
print("=== 缺失值统计 ===")
print(f"  out 缺失: {np.isnan(outage).sum()} / {outage.size}")
print(f"  tracked 缺失: {np.isnan(tracked).sum()} / {tracked.size}")

weather_nan_per_feat = np.isnan(weather.reshape(-1, F)).sum(axis=0)
nan_df = pd.DataFrame({'feature': weather_features, 'nan_count': weather_nan_per_feat})
nan_df['nan_pct'] = nan_df['nan_count'] / (T * L) * 100
nan_with_missing = nan_df[nan_df['nan_count'] > 0].sort_values('nan_count', ascending=False)

if len(nan_with_missing) > 0:
    print(f"\n  有缺失值的天气特征 ({len(nan_with_missing)} 个):")
    print(nan_with_missing.to_string(index=False))
else:
    print("\n  所有天气特征均无缺失值")

---
# Task C: 通用特征构造

**目标**：构造时间特征、滞后特征、滑动窗口统计，封装成函数大家共享。

这些特征无论后续用什么模型（树模型、LSTM、SARIMAX）都有用。

In [ ]:
# Task C: 通用特征构造函数
# 注意：所有特征只用过去的信息，不能用未来数据（防止数据泄漏）

def build_features(ds, selected_weather_features=None):
    """
    从 xarray dataset 构造特征 DataFrame。
    
    参数:
        ds: xarray Dataset，包含 out, tracked, weather, timestamp, location
        selected_weather_features: 筛选后的天气特征名列表 (None = 用全部)
    
    返回:
        df: DataFrame，每行是一个 (timestamp, location) 的观测，包含所有构造的特征
    """
    timestamps = pd.to_datetime(ds.timestamp.values)
    locations = list(ds.location.values)
    all_features = list(ds.feature.values)
    
    out = ds.out.transpose("timestamp", "location").values.astype(float)
    trk = ds.tracked.transpose("timestamp", "location").values.astype(float)
    w = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)
    
    T, L = out.shape
    records = []
    
    for li in range(L):
        loc = locations[li]
        df_loc = pd.DataFrame({
            'timestamp': timestamps,
            'location': loc,
            'out': out[:, li],
            'tracked': trk[:, li],
        })
        
        # ====== 1. 时间特征 ======
        df_loc['hour'] = timestamps.hour
        df_loc['day_of_week'] = timestamps.dayofweek
        df_loc['month'] = timestamps.month
        # sin/cos 编码（让 23点和0点相近）
        df_loc['hour_sin'] = np.sin(2 * np.pi * df_loc['hour'] / 24)
        df_loc['hour_cos'] = np.cos(2 * np.pi * df_loc['hour'] / 24)
        df_loc['dow_sin'] = np.sin(2 * np.pi * df_loc['day_of_week'] / 7)
        df_loc['dow_cos'] = np.cos(2 * np.pi * df_loc['day_of_week'] / 7)
        
        # ====== 2. 停电滞后特征 ======
        for lag in [1, 3, 6, 12, 24]:
            df_loc[f'out_lag_{lag}h'] = df_loc['out'].shift(lag)
        
        # ====== 3. 停电滑动窗口统计 ======
        for window in [6, 12, 24]:
            rolling = df_loc['out'].shift(1).rolling(window, min_periods=1)
            df_loc[f'out_roll_mean_{window}h'] = rolling.mean()
            df_loc[f'out_roll_max_{window}h'] = rolling.max()
            df_loc[f'out_roll_std_{window}h'] = rolling.std().fillna(0)
        
        # ====== 4. 天气特征（筛选后的）======
        if selected_weather_features is not None:
            feat_indices = [all_features.index(f) for f in selected_weather_features]
            feat_names = selected_weather_features
        else:
            feat_indices = list(range(len(all_features)))
            feat_names = all_features
        
        for fi, fname in zip(feat_indices, feat_names):
            df_loc[f'w_{fname}'] = w[:, li, fi]
        
        # ====== 5. 关键天气的滑动统计（可选，只对 top 特征做）======
        # 这里先跳过，在确定 top 特征后再补充
        
        records.append(df_loc)
    
    df = pd.concat(records, ignore_index=True)
    return df

print("build_features() 函数定义完成")
print("用法: df = build_features(ds, selected_weather_features=['gust', 't2m', 'cape', ...])")
print("如果不传 selected_weather_features，会用全部 109 个天气变量")

In [ ]:
# Task C: 试运行 — 用筛选后的 top 特征构造
# 先用 B1 得到的 top 15 特征测试一下（跑完 B1 后这里才有值）
try:
    top15_features = list(corr_series.head(15).index)
    print(f"使用 top 15 天气特征: {top15_features}\n")
    
    df = build_features(ds, selected_weather_features=top15_features)
    
    print(f"构造后的 DataFrame:")
    print(f"  Shape: {df.shape}  ({df.shape[0]:,} 行 x {df.shape[1]} 列)")
    print(f"  列名: {list(df.columns)}")
    print(f"\n  前几行预览:")
    print(df.head(3).to_string())
    
    # 检查 NaN（lag 特征在前几行会有 NaN，这是正常的）
    nan_counts = df.isnull().sum()
    print(f"\n  有 NaN 的列 (lag/rolling 的前几行会有，正常):")
    print(nan_counts[nan_counts > 0])
except NameError:
    print("请先运行 Task B 的代码得到 corr_series")

---
# Task D: Baseline 复现 & 评估框架

**目标**：跑通 baseline，记录 RMSE，搭建统一评估函数。

## D1: 评估函数

统一的评估函数，所有模型都用这个来算分，方便对比。

In [ ]:
# D1: 统一评估函数
def evaluate_model(y_true, y_pred, locations, model_name="Model"):
    """
    评估模型预测结果。
    
    参数:
        y_true: (T, L) 真实停电数，T=时间步数, L=县数量
        y_pred: (T, L) 预测停电数
        locations: 县名列表
        model_name: 模型名称（用于打印）
    
    返回:
        overall_rmse: 全局平均 RMSE
        county_rmses: dict, {county: rmse}
    """
    county_rmses = {}
    for li, loc in enumerate(locations):
        diff = y_true[:, li] - y_pred[:, li]
        county_rmses[loc] = np.sqrt(np.mean(diff**2))
    
    overall_rmse = np.mean(list(county_rmses.values()))
    
    print(f"=== {model_name} ===")
    print(f"  Overall RMSE (县均): {overall_rmse:.4f}")
    
    # Top 5 最差的县
    sorted_counties = sorted(county_rmses.items(), key=lambda x: x[1], reverse=True)
    print(f"  RMSE 最高的 5 个县:")
    for loc, rmse in sorted_counties[:5]:
        print(f"    {loc}: {rmse:.4f}")
    
    return overall_rmse, county_rmses


def compare_models(results_dict):
    """
    对比多个模型的结果。
    
    参数:
        results_dict: {model_name: overall_rmse}
    """
    print("\n" + "="*50)
    print("模型对比")
    print("="*50)
    df = pd.DataFrame([
        {'Model': name, 'RMSE': rmse} 
        for name, rmse in sorted(results_dict.items(), key=lambda x: x[1])
    ])
    print(df.to_string(index=False))
    print(f"\n当前最优: {df.iloc[0]['Model']} (RMSE={df.iloc[0]['RMSE']:.4f})")

print("evaluate_model() 和 compare_models() 函数定义完成")

## D2: Persistence Baseline

最简单的 baseline：用每个县最后一个观测值作为未来所有时刻的预测。这是"什么都不做"的下限。

In [ ]:
# D2: Persistence baseline + 验证集评估
# 数据划分：前 80% 训练，后 20% 验证（和 demo 一致）
VALIDATION_SPLIT = 0.2
split_idx = int(T * (1 - VALIDATION_SPLIT))

train_outage = outage[:split_idx, :]   # (T_train, L)
val_outage = outage[split_idx:, :]     # (T_val, L)
val_timestamps = timestamps[split_idx:]

print(f"训练集: {split_idx} 小时 ({timestamps[0].date()} ~ {timestamps[split_idx-1].date()})")
print(f"验证集: {T - split_idx} 小时 ({timestamps[split_idx].date()} ~ {timestamps[-1].date()})")

# --- Persistence baseline ---
# 24h: 用训练集最后一个值重复 24 次
# 48h: 用训练集最后一个值重复 48 次

val_24h = val_outage[:24, :]  # 验证集前 24 小时
val_48h = val_outage[:48, :]  # 验证集前 48 小时

last_val = train_outage[-1, :]  # 训练集最后一个时刻的停电数 (L,)

persistence_24h = np.tile(last_val, (24, 1))  # (24, L)
persistence_48h = np.tile(last_val, (48, 1))  # (48, L)

# 评估
results = {}
rmse_24, _ = evaluate_model(val_24h, persistence_24h, locations, "Persistence 24h")
rmse_48, _ = evaluate_model(val_48h, persistence_48h, locations, "Persistence 48h")
results['Persistence_24h'] = rmse_24
results['Persistence_48h'] = rmse_48

# --- Historical Average baseline ---
# 用训练集每个县的全局平均值作为预测
hist_avg = train_outage.mean(axis=0)  # (L,)
hist_avg_24h = np.tile(hist_avg, (24, 1))
hist_avg_48h = np.tile(hist_avg, (48, 1))

print()
rmse_24_ha, _ = evaluate_model(val_24h, hist_avg_24h, locations, "Historical Average 24h")
rmse_48_ha, _ = evaluate_model(val_48h, hist_avg_48h, locations, "Historical Average 48h")
results['HistAvg_24h'] = rmse_24_ha
results['HistAvg_48h'] = rmse_48_ha

print("\n--- 后续: 跑完 demo.ipynb 的 SARIMAX 和 Seq2Seq 后把它们的 RMSE 也填进来 ---")
# results['SARIMAX_24h'] = xxx
# results['SARIMAX_48h'] = xxx
# results['Seq2Seq_24h'] = xxx
# results['Seq2Seq_48h'] = xxx
# compare_models(results)

---
# Phase 1 总结 & 下一步

跑完这个 notebook 后，你应该知道：

1. **数据特点**：停电数据极度稀疏，极端值由天气事件驱动
2. **重要特征**：从 109 个天气变量筛选到了 ~15 个最相关的
3. **特征工程**：`build_features()` 函数可以直接给后续模型用
4. **Baseline 水平**：Persistence / Historical Average / SARIMAX / Seq2Seq 的 RMSE 已知

**Phase 2 的方向**：
- 尝试更强的模型（XGBoost, LightGBM, Transformer 等）
- 用 `build_features()` 的输出作为起点，根据模型特点进一步调整
- 模型集成（Ensemble）
- Part II: 基于预测结果做发电机部署决策